# Evaluating cpu-time for each version of gl(r), built over time.


## File: `gl_computing_speed.ipynb`  
**Project:** HHG-SaDAS  

## Code Description  
This notebook contains various implementations of partial wave calculations developed over time, with an emphasis on improving efficiency and reducing computational cost. Different formulations of the radial and angular parts of the wavefunction are tested and compared, along with their numerical performance.  

---

**Author:** Siddhartha Mithiya  
**Affiliation:** Indian Institute of Technology (IIT) Mandi  
**License:** MIT License  
**Repository:** [https://github.com/Siddhartha-Acad/HHG-SaDAS.git](https://github.com/Siddhartha-Acad/HHG-SaDAS.git)  

---

## Notes  
- Compares different function implementations for accuracy, efficiency, and runtime scaling.  
- Demonstrates importance of vectorization, memory preallocation, and avoiding redundant computations in scientific code.  
- This notebook is part of the HHG-SaDAS package, developed during my MS(R) thesis:  
  *"Higher-Order Harmonic Generation and Harmonic-Power Enhancement in Noble-Gas Atoms Confined Inside C60".*  


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import legendre

In [2]:
def f(x):
    alpha = 2 * L_map / r_max
    return L_map * (1 + x) / (1 - x + alpha)

def f_p(x):
    alpha = 2 * L_map / r_max
    return L_map * (alpha + 2) / (1 - x + alpha)**2

In [3]:
n, l = 1, 0
N, L_map, r_max = 200, 80, 200                         # Radial Mapping parameters
L = 50                                                 # Angular grid points (L+1)
l_max = 50                                             # Total number of partial waves (l_max+1)

In [4]:
colloc_file = 'Algo-3_N=200_AnaDeriv_collocation_points.txt'     # importing collocation point data: (x_i).
colloc_file = rf'E:\Python_programs\HHG\GPSM\Collocation_points\AnaDeriv_Colloc_pt\{colloc_file}'
colloc_pt = np.loadtxt(colloc_file, skiprows=1, usecols=0)       # Gauss-Lobatto collocation points

r = f(colloc_pt)                                                 # nonlinear radial grid: r(x)

In [5]:
roots, weights = np.polynomial.legendre.leggauss(L+1)  # Gauss-Legendre Quadrature.
theta_k = np.arccos(roots)                             # Gauss-Legendre angular collocation points
len_r = len(r); len_k = len(weights)

In [6]:
A_file = 'H_GPSM_States__l=0_nos=10_N=200_rmax=200_Lmap=80.xlsx'
A_file = rf'E:\Python_programs\HHG\GPSM\GPSM_Y_l0\free_H\File_Generator\{A_file}'
A_data = pd.read_excel(A_file, header=None, skiprows=1).to_numpy().T                 # wavefunction data.

A_r = A_data[1:][n-1]
A_m = np.empty((len(theta_k), len(r)))
norm_fact = np.sqrt(N * (N + 1) / 2)
r_m, theta_m = np.meshgrid(r, theta_k)

for k in range(len(theta_k)):
    A_m[k] = A_r * np.sqrt((2*l+1) / (4*np.pi)) * legendre(l)(np.cos(theta_k[k]))

In [7]:
def gl_v1(Psi_t):
    """
    Compute partial waves g_l(r) for a given wavefunction using Gauss–Legendre quadrature.

    This function calculates the partial wave components g_l(r) of a wavefunction ψ(θ, r)
    by projecting onto Legendre polynomials evaluated at Gauss–Legendre nodes.

    Parameters
    ----------
    Psi_t : ndarray, shape (len_k, len_r)
        Complex-valued wavefunction evaluated at Gauss–Legendre angular points θ_k
        and radial grid points r_i. Expected ordering:
            Psi_t[k, i] = ψ(θ_k, r_i).

    Returns
    -------
    gl_arr : list of ndarray
        Partial wave components g_l(r), one array for each angular momentum index
        l = 0,...,l_max. Each array has shape (len_r,) and is complex-valued.

    Notes
    -----
    - Uses global variables: len_k, len_r, l_max, roots, and weights.
    - Legendre polynomials P_l(cosθ) are evaluated at the Gauss–Legendre nodes.
    - The projection is performed using the Gauss–Legendre quadrature scheme.
    """

    
    gl_arr = []
    legendre_vals = np.array([[legendre(l_index)(root) for root in roots] for l_index in range(l_max+1)])
    for l_ind in range(l_max+1):
        g_l = np.zeros(len_r, dtype=np.complex128)
        psi_vals = np.array([[Psi_t[k][i] for k in range(len_k)] for i in range(len_r)])
        for i in range(len_r):
            g_l[i] = np.sum(weights * legendre_vals[l_ind, :] * psi_vals[i, :])
        gl_arr.append(g_l * np.sqrt(np.pi * (2 * l_ind + 1)))
    return gl_arr


In [8]:
def gl_v2(Psi_t):
    """
    Compute the partial wave projections g_l(r) of a given wavefunction using Gauss-Legendre quadrature.

    Parameters
    ----------
    Psi_t : ndarray of shape (len_k, len_r)
        The full angular-radial wavefunction ψ(θ, r) at a fixed time. It must include
        function values at the Gauss-Legendre quadrature points θ_k. This function expects:
            Psi_t[k, i] = ψ(θ_k, r_i)

    Returns
    -------
    gl_arr : list of ndarray, each of shape (len_r,)
        A list of partial wave projections g_l(r) for l = 0 to l_max. Each element is a real or
        complex-valued array representing the radial component of the wavefunction for the
        corresponding angular momentum quantum number l.

    Notes
    -----
    - The projection is computed using:
        g_l(r) = √[π (2l + 1)] × ∑_k w_k P_l(cosθ_k) ψ(θ_k, r)
      where the sum is evaluated using Gauss-Legendre quadrature weights `weights` and
      Legendre polynomial values `legendre_vals[l_ind, :]`.

    - This implementation constructs an intermediate (len_r × len_k) array and stores
      results in a Python list, making it less memory efficient for repeated calls.
    """

    gl_arr = []
    psi_vals = np.array([[Psi_t[k][i] for k in range(len_k)] for i in range(len_r)])
    for l_ind in range(l_max+1):
        g_l = (weights * legendre_vals[l_ind, :]) @ psi_vals.T
        gl_arr.append(g_l * np.sqrt(np.pi * (2*l_ind + 1)))
    return gl_arr

legendre_vals = np.array([[legendre(l_index)(root) for root in roots] for l_index in range(l_max+1)])     # used in gl(r)

In [9]:
def gl_v3(Psi_t):
    """
    Compute partial wave projections g_l(r) from a given wavefunction ψ(θ, r) using
    an efficient, memory-optimized approach with Gauss-Legendre quadrature.

    Parameters
    ----------
    Psi_t : ndarray, shape (len_k, len_r)
        Wavefunction values ψ(θ_k, r_i) evaluated at Gauss–Legendre nodes θ_k and
        radial grid points r_i. Expected ordering:
            Psi_t[k, i] = ψ(θ_k, r_i).

    Returns
    -------
    gl_arr : list of ndarray, each of shape (len_r,)
        A list of partial wave projections g_l(r) for l = 0 to l_max. Each element is
        a NumPy array representing the radial component of the wavefunction for the
        corresponding angular momentum quantum number l.

    Notes
    -----
    - This version avoids avoids explicitly constructing the (len_r × len_k) matrix,
      reducing memory usage compared to gl_v1.
    - The projection is computed as:
        g_l(r) = √[π (2l + 1)] × ∑_k w_k P_l(cosθ_k) ψ(θ_k, r)
      where the sum is evaluated efficiently using `np.tensordot`.
    - The result is returned as a list, where each entry corresponds to a different l.
    """

    gl_arr = []
    for l_ind in range(l_max+1):
        proj = np.tensordot(weights * legendre_vals[l_ind, :], Psi_t, axes=([0], [0]))
        gl_arr.append(proj * np.sqrt(np.pi * (2*l_ind + 1)))
    return gl_arr


In [10]:
def gl_v4(Psi_t):
    """
    Efficient and memory-optimized computation of partial wave projections g_l(r)
    from a given angular-radial wavefunction using Gauss–Legendre quadrature.

    Parameters
    ----------
    Psi_t : ndarray, shape (len_k, len_r)
        Wavefunction values ψ(θ_k, r_i) evaluated at Gauss–Legendre nodes θ_k and
        radial grid points r_i. Expected ordering:
            Psi_t[k, i] = ψ(θ_k, r_i).

    Returns
    -------
    gl_arr : ndarray, shape (l_max+1, len_r)
        Array where row l contains the radial projection g_l(r) for angular
        momentum index l = 0,...,l_max.

    Notes
    -----
    - Uses preallocated arrays instead of Python lists for improved performance
      and cache efficiency.
    - Projection is computed as
          g_l(r) = √[π (2l + 1)] × ∑_k w_k P_l(cosθ_k) ψ(θ_k, r),
      with Gauss–Legendre weights `weights` and precomputed Legendre values
      `legendre_vals[l, k]`.
    - Requires global variables: len_k, len_r, l_max, weights, legendre_vals.
    - Suitable for repeated, high-performance applications (e.g. time evolution).
    """


    gl_arr = np.empty((l_max+1, len_r), dtype=np.complex128)
    for l_ind in range(l_max+1):
        gl_arr[l_ind] = (np.sqrt(np.pi * (2*l_ind + 1)) *
                         np.tensordot(weights*legendre_vals[l_ind, :], Psi_t, axes=([0], [0])))
    return gl_arr


In [11]:
def gl_v5(Psi_t, gl_arr=None):
    """
    Compute partial wave projections g_l(r) from a given angular-radial wavefunction
    using Gauss–Legendre quadrature, with support for memory reuse.

    Parameters
    ----------
    Psi_t : ndarray, shape (len_k, len_r)
        Wavefunction values ψ(θ_k, r_i) evaluated at Gauss–Legendre nodes θ_k and
        radial grid points r_i. Expected ordering:
            Psi_t[k, i] = ψ(θ_k, r_i).

    gl_arr : ndarray, shape (l_max+1, len_r), optional
        Optional output array to store the result. If provided, it will be filled
        in-place and returned, avoiding repeated memory allocations. If None,
        a new array is allocated.

    Returns
    -------
    gl_arr : ndarray, shape (l_max+1, len_r)
        Radial partial wave projections g_l(r). Each row corresponds to angular
        momentum index l = 0,...,l_max.

    Notes
    -----
    - Designed for performance-critical contexts (e.g., time evolution loops),
      where avoiding memory allocations improves efficiency.
    - Projection is computed as
          g_l(r) = √[π (2l + 1)] × ∑_k w_k P_l(cosθ_k) ψ(θ_k, r),
      using Gauss–Legendre quadrature with weights `weights` and precomputed
      Legendre values `legendre_vals[l, k]`.

    Example
    -------
    >>> gl_empty = np.empty((l_max+1, len_r), dtype=np.complex128)
    >>> for t in range(num_steps):
    ...     update_Psi_t(...)
    ...     gl_vals = gl_v5(Psi_t, gl_empty)  # memory is reused each time
    """


    if gl_arr is None:
        gl_arr = np.empty((l_max+1, len_r), dtype=np.complex128)

    for l_ind in range(l_max+1):
        gl_arr[l_ind] = (np.sqrt(np.pi * (2*l_ind + 1)) *
                         np.tensordot(weights * legendre_vals[l_ind, :], Psi_t, axes=([0], [0])))
    return gl_arr

In [12]:
%timeit gl_v1(A_m)

743 ms ± 26.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [13]:
%timeit gl_v2(A_m)

5.28 ms ± 102 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [14]:
%timeit gl_v3(A_m)

3.06 ms ± 143 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [15]:
%timeit gl_v4(A_m)

3.13 ms ± 140 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [16]:
gl_empty = np.empty((l_max + 1, len_r), dtype=np.complex128)

In [17]:
%timeit gl_v5(A_m, gl_empty)

3.05 ms ± 93.7 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
